# Tutorial: Realism Audit and Gate Triage

Detailed step-by-step workflow notebook for this SD-dMFA repository.


## Audience, Prerequisites, Outcomes

**Audience**
- Model reviewers validating scenario realism and stability gates.

**Prerequisites**
- Python 3.11+ environment for this repo.
- `pip install -e ".[dev]"` completed.
- Notebook executed from repository root or a subfolder.

**Outcomes**
- Run realism audit across config sets.
- Inspect gate pass/fail outcomes and details.
- Build actionable triage notes per failing gate.


## Outline

1. Run audit script
2. Load audit CSV and summary markdown
3. Filter failing gates
4. Map failures to intervention candidates
5. Generate triage export table


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
from pathlib import Path
from textwrap import dedent

import pandas as pd
import matplotlib.pyplot as plt

try:
    from crm_model.common.io import load_run_config
except Exception:
    load_run_config = None

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)


In [ ]:
DRY_RUN = False
RUN_HEAVY = False
RUN_PLOTS = False
RUN_CALIBRATION = False
RUN_AUDIT = False

CONFIG = "configs/runs/mvp.yml"
EXAMPLE_VARIANT = "baseline"


In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "configs").exists() and (cand / "src").exists():
            return cand
    raise RuntimeError("Could not locate repo root from current working directory.")


def sh(cmd: str, *, cwd: Path, check: bool = True) -> subprocess.CompletedProcess | None:
    print(f"$ {cmd}")
    if DRY_RUN:
        return None
    cp = subprocess.run(cmd, cwd=str(cwd), shell=True, text=True, capture_output=True)
    if cp.stdout.strip():
        print(cp.stdout)
    if cp.stderr.strip():
        print(cp.stderr)
    if check and cp.returncode != 0:
        raise RuntimeError(f"Command failed ({cp.returncode}): {cmd}")
    return cp


def latest_dir(base: Path) -> Path | None:
    if not base.exists():
        return None
    cands = [p for p in base.iterdir() if p.is_dir() and p.name != "_archive"]
    return sorted(cands)[-1] if cands else None


def load_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        print(f"Missing: {path}")
        return pd.DataFrame()
    return pd.read_csv(path)


REPO = find_repo_root()
CONFIG_PATH = (REPO / CONFIG).resolve()
CONFIG_STEM = CONFIG_PATH.stem
print("Repo:", REPO)
print("Config:", CONFIG_PATH)


## Step 1: Execute realism audit


In [ ]:
if RUN_AUDIT:
    _ = sh("python scripts/analysis/audit_scenario_realism.py --config configs/runs/mvp.yml --config configs/runs/r-strategies.yml", cwd=REPO, check=False)
else:
    print("Set RUN_AUDIT=True to execute")


## Step 2: Load latest audit artifacts


In [ ]:
audit_dir = REPO / "outputs" / "analysis" / "scenario_realism" / "latest"
audit_csv = audit_dir / "realism_audit.csv"
audit_md = audit_dir / "realism_summary.md"

audit = load_csv(audit_csv)
print("audit rows:", len(audit))
if audit_md.exists():
    print(audit_md.read_text(encoding="utf-8")[:1200])


## Step 3: Review failing gates


In [ ]:
if not audit.empty:
    failing = audit[audit["passed"] == False].copy()
    display(failing)


## Step 4: Map gate failures to candidate actions


In [ ]:
playbook = {
    "baseline_chronic_under_service": "Adjust baseline supply/demand/scarcity alignment and check data baselines.",
    "strategic_reserve_iterations": "Dampen strategic reserve gains/targets and strategic smoothing path.",
    "r36_monotonicity": "Recalibrate r36 low/med/high ladder values and ramp separations.",
    "r_strategy_bottleneck_activation": "Increase constrained-capacity windows or demand pressure in >=1 r-variant.",
    "all_converged": "Inspect convergence metric components and iteration traces per slice.",
}
if not audit.empty:
    triage = audit.copy()
    triage["candidate_action"] = triage["gate_id"].map(playbook).fillna("Define model-specific corrective action")
    display(triage)


## Step 5: Export triage table


In [ ]:
if not audit.empty:
    triage_out = REPO / "outputs" / "analysis" / "scenario_realism" / "latest" / "realism_triage.csv"
    triage.to_csv(triage_out, index=False)
    print("Wrote", triage_out)


## Pitfalls

- Audit script returns non-zero exit code when any gate fails; this is expected behavior.
- Always inspect detailed diagnostics before tuning parameters.


## Exercises

1. Repeat this workflow with `CONFIG=configs/runs/r-strategies.yml`.
2. Record one thing that changed and why.
3. Add one guardrail/check specific to your team workflow.


In [ ]:
# Exercise answer scaffold
pass
